In [ ]:
import paddle
from paddle.nn import Linear
import paddle.nn.functional as F
import numpy as np
import pandas as pd
import os
import random

def loaddata():
	boston = "http://lib.stat.cmu.edu/datasets/boston"
	raw_df = pd.read_csv(boston, sep=r"\s+", skiprows=22, header=None)
	data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :3]])
	# target = raw_df.values[1::2, 2]
	feature_names = [
		'CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE',
		'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT'
	]
	feature_num = len(feature_names)
	print("feature_num = ",feature_num)

	ratio = 0.8
	offset = int(data.shape[0]*ratio)
	training  = data[:offset]
	maximums,minimums = training.max(axis=0),training.min(axis=0)
	for i in range(feature_num):
		data[:,i] = (data[:,i] - minimums[i])/(maximums[i]-minimums[i])

	training = data[:offset]
	test = data[offset:]
	return training,test

class regressor(paddle.nn.Layer): # regressor:[统]回归量
	def __init__(self):
		# super() 函数是用于调用父类(超类)的一个方法
		super(regressor,self).__init__()
		# 定义一层全连接层，输入维度是13，输出维度是1
		self.fc = Linear(in_features = 13,out_features = 1)

	def forward(self,inputs):
		x=self.fc(inputs)
		return x



model = regressor()
model.train()
train_data, test_data = loaddata()
# 定义优化算法，采用随机梯度下降SGD
# 学习率设置为0.01
opt = paddle.optimizer.SGD(learning_rate=0.01,parameters=model.parameters())



epoch_num = 20   # 设置模型训练轮次
batch_size = 10  # 设置批大小，即一次模型训练使用的样本数量

# 定义模型训练轮次epoch（外层循环）
for epoch_id in range(epoch_num):
    # 在每轮迭代开始之前，对训练集数据进行样本乱序
    np.random.shuffle(train_data)
    # 对训练集数据进行拆分，batch_size设置为10
    mini_batches = [train_data[k:k+batch_size] for k in range(0, len(train_data), batch_size)]
    # 定义模型训练（内层循环）
    for iter_id, mini_batch in enumerate(mini_batches):
        x = np.array(mini_batch[:, :-1]) # 将当前批的房价影响因素的数据转换为np.array格式
        y = np.array(mini_batch[:, -1:]) # 将当前批的标签数据（真实房价）转换为np.array格式
        # 将np.array格式的数据转为张量tensor格式
        house_features = paddle.to_tensor(x, dtype='float32')
        prices = paddle.to_tensor(y, dtype='float32')
        
        # 前向计算
        predicts = model(house_features)

        # 计算损失，损失函数采用平方误差square_error_cost
        loss = F.square_error_cost(predicts, label=prices)
        avg_loss = paddle.mean(loss)
        if iter_id%20==0:
            print("epoch: {}, iter: {}, loss is: {}".format(epoch_id, iter_id, avg_loss.numpy()))
        
        # 反向传播，计算每层参数的梯度值
        avg_loss.backward()
        # 更新参数，根据设置好的学习率迭代一步
        opt.step()
        # 清空梯度变量，进行下一轮计算
        opt.clear_grad()

paddle.save(model.state_dict(), 'LR_model.pdparams')

feature_num =  13
epoch: 0, iter: 0, loss is: 601.5360107421875
epoch: 0, iter: 20, loss is: 68.7932357788086
epoch: 0, iter: 40, loss is: 120.79659271240234
epoch: 1, iter: 0, loss is: 93.11653137207031
epoch: 1, iter: 20, loss is: 50.91412353515625
epoch: 1, iter: 40, loss is: 68.86235809326172
epoch: 2, iter: 0, loss is: 161.2272186279297
epoch: 2, iter: 20, loss is: 92.10726928710938
epoch: 2, iter: 40, loss is: 104.82345581054688
epoch: 3, iter: 0, loss is: 42.33143997192383
epoch: 3, iter: 20, loss is: 29.577106475830078
epoch: 3, iter: 40, loss is: 56.05699920654297
epoch: 4, iter: 0, loss is: 68.72915649414062
epoch: 4, iter: 20, loss is: 21.011219024658203
epoch: 4, iter: 40, loss is: 19.708602905273438
epoch: 5, iter: 0, loss is: 56.3580436706543
epoch: 5, iter: 20, loss is: 76.29157257080078
epoch: 5, iter: 40, loss is: 20.615299224853516
epoch: 6, iter: 0, loss is: 38.80400848388672
epoch: 6, iter: 20, loss is: 46.86091995239258
epoch: 6, iter: 40, loss is: 30.1257934570312

In [10]:
import paddle
paddle.set_default_dtype("float32")

train_dataset = paddle.text.datasets.UCIHousing(mode="train")
eval_dataset = paddle.text.datasets.UCIHousing(mode="test")
model = paddle.Model(regressor())
model.prepare(paddle.optimizer.SGD(learning_rate=0.005,parameters=model.parameters()),
			paddle.nn.MSELoss())
model.fit(train_dataset,eval_dataset,epochs=10,batch_size=10,verbose=1) #verbose:冗长的;啰嗦的;累赘的



The loss value printed in the log is the current step, and the metric is the average value of previous steps.
Epoch 1/10
step 41/41 [==============================] - loss: 274.0028 - 1ms/step                                
Eval begin...
step 11/11 [==============================] - loss: 95.5438 - 554us/step                               
Eval samples: 102
Epoch 2/10
step 41/41 [==============================] - loss: 49.8628 - 679us/step                               
Eval begin...
step 11/11 [==============================] - loss: 34.2774 - 386us/step                               
Eval samples: 102
Epoch 3/10
step 41/41 [==============================] - loss: 18.7012 - 640us/step                              
Eval begin...
step 11/11 [==============================] - loss: 25.5280 - 490us/step                              
Eval samples: 102
Epoch 4/10
step 41/41 [==============================] - loss: 24.4526 - 588us/step                              
Eval begin...
step 11/11 

c:\Users\chenx\.conda\envs\buy_oranges\Lib\site-packages\paddle\hapi\model.py:1246: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach(), rather than paddle.to_tensor(sourceTensor).
  labels = [paddle.to_tensor(l) for l in to_list(labels)]


step 41/41 [==============================] - loss: 11.7728 - 747us/step                              
Eval begin...
step 11/11 [==============================] - loss: 42.0701 - 74us/step                             
Eval samples: 102
Epoch 6/10
step 41/41 [==============================] - loss: 26.7115 - 832us/step                               
Eval begin...
step 11/11 [==============================] - loss: 50.5526 - 392us/step                              
Eval samples: 102
Epoch 7/10
step 41/41 [==============================] - loss: 19.6534 - 869us/step                              
Eval begin...
step 11/11 [==============================] - loss: 56.3246 - 569us/step                             
Eval samples: 102
Epoch 8/10
step 41/41 [==============================] - loss: 22.4057 - 660us/step                              
Eval begin...
step 11/11 [==============================] - loss: 59.8676 - 583us/step                             
Eval samples: 102
Epoch 9/10
step 41